<a href="https://colab.research.google.com/github/rodrigopozza/QUANTUM_QISKIT/blob/main/TCEzipBanco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import base64
import json
import time
import io
import zipfile
import psycopg2  # Conector do PostgreSQL
from psycopg2 import extras
import requests
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET

# 1. Configurações do TCE-PR (Paranavaí 2026)
CODIGO_MUNICIPIO = "18402"
NOME_MUNICIPIO = "PARANAVAÍ"
ANO = "2026"

# 2. Configurações de Conexão do PostgreSQL (Ajuste com os seus dados)
DB_CONFIG = {
    "host": "localhost",
    "database": "seu_banco_de_dados",
    "user": "seu_usuario",
    "password": "sua_senha",
    "port": 5432
}

def conectar_banco():
    """Retorna uma conexão com o PostgreSQL"""
    return psycopg2.connect(**DB_CONFIG)

def configurar_banco():
    """Cria a tabela no PostgreSQL se não existir"""
    conn = conectar_banco()
    cursor = conn.cursor()

    # Criando a tabela. Usamos o tipo TEXT para o XML puro,
    # mas se o seu Postgres estiver configurado, pode usar o tipo nativo XML.
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS dados_tce_pr (
            id SERIAL PRIMARY KEY,
            municipio VARCHAR(100),
            ano VARCHAR(4),
            nome_arquivo VARCHAR(255),
            entidade VARCHAR(255),
            valor NUMERIC(15, 2),
            data_registro DATE,
            xml_completo TEXT,
            capturado_em TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    conn.commit()
    cursor.close()
    conn.close()
    print("[V] Estrutura do PostgreSQL verificada/criada.")

def gerar_url_consulta():
    dados_busca = {"cdMunicipio": CODIGO_MUNICIPIO, "nrAno": ANO, "municipio": NOME_MUNICIPIO}
    json_string = json.dumps(dados_busca, ensure_ascii=False)
    base64_string = base64.b64encode(json_string.encode('utf-8')).decode('utf-8')
    return f"https://pit.tce.pr.gov.br/Dados/DadosConsulta/Consulta/?f={base64_string}"

def processar_e_salvar_no_postgres(conteudo_xml, nome_arquivo):
    """Lê o XML da memória e faz o INSERT direto no PostgreSQL"""
    conn = conectar_banco()
    cursor = conn.cursor()

    try:
        # Transforma os bytes em uma árvore XML manipulável
        root = ET.fromstring(conteudo_xml)

        # --- Mapeamento das Tags (Ajuste conforme os nós do XML real do TCE) ---
        entidade = root.find('.//entidade').text if root.find('.//entidade') is not None else "Não informada"
        valor = float(root.find('.//valor').text) if root.find('.//valor') is not None else 0.0
        data = root.find('.//data').text if root.find('.//data') is not None else "2026-01-01"

        # Decodifica os bytes do XML para string (UTF-8) para salvar no banco
        xml_string = conteudo_xml.decode('utf-8', errors='ignore')

        # Comando SQL para o PostgreSQL (usando placeholders %s)
        sql = '''
            INSERT INTO dados_tce_pr (municipio, ano, nome_arquivo, entity, valor, data_registro, xml_completo)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        '''

        cursor.execute(sql, (NOME_MUNICIPIO, ANO, nome_arquivo, entidade, valor, data, xml_string))
        conn.commit()
        print(f"   [V] Arquivo '{nome_arquivo}' inserido no Postgres!")

    except ET.ParseError:
        print(f"   [-] Erro de estrutura XML no arquivo '{nome_arquivo}'.")
    except Exception as e:
        print(f"   [-] Erro ao inserir no PostgreSQL para o arquivo '{nome_arquivo}': {e}")
        conn.rollback() # Cancela a transação em caso de falha
    finally:
        cursor.close()
        conn.close()

def executar_automacao_postgres():
    session = requests.Session()
    session.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})

    print("[+] Conectando ao portal do TCE-PR...")
    url_consulta = gerar_url_consulta()
    resposta = session.get(url_consulta, timeout=15)

    if resposta.status_code != 200:
        print("[-] Falha ao carregar o portal.")
        return

    soup = BeautifulSoup(resposta.text, 'html.parser')
    links_arquivos = []
    for tag_a in soup.find_all('a', href=True):
        href = tag_a['href']
        if any(termo in href.lower() for termo in ['.xml', '.zip', 'download', 'obterarquivo']):
            if href.startswith('/'): href = f"https://pit.tce.pr.gov.br{href}"
            links_arquivos.append(href)

    print(f"[+] Encontrados {len(links_arquivos)} pacotes para download.")

    for indice, url_download in enumerate(links_arquivos):
        print(f" -> Processando pacote {indice + 1}...")
        res_file = session.get(url_download, timeout=45)

        if res_file.status_code == 200:
            conteudo_memoria = io.BytesIO(res_file.content)

            # Se for um arquivo ZIP, abre e extrai na RAM
            if zipfile.is_zipfile(conteudo_memoria):
                with zipfile.ZipFile(conteudo_memoria) as z:
                    for nome_xml in z.namelist():
                        if nome_xml.lower().endswith('.xml'):
                            xml_bytes = z.read(nome_xml)
                            processar_e_salvar_no_postgres(xml_bytes, nome_xml)
            else:
                # Se for o XML direto
                processar_e_salvar_no_postgres(res_file.content, f"direto_{indice+1}.xml")

        time.sleep(1)

if __name__ == "__main__":
    try:
        configurar_banco()
        executar_automacao_postgres()
        print("\n[FIM] Automação concluída com sucesso no PostgreSQL!")
    except Exception as e:
        print(f"\n[-] Erro de conexão com o banco de dados: {e}")


[-] Erro de conexão com o banco de dados: connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?

